In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings_model = HuggingFaceEmbeddings(
    model_name="./model/bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    },  # 输出归一化向量，更适合余弦相似度计算
)

vectorstore = Chroma(
    embedding_function=embeddings_model,
    persist_directory="./vectorstore" #
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import HumanMessage, AIMessage
#LCEL方式来构建检索生成过程
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)

#ChatPromptTemplate
prompt = PromptTemplate(
    input_variables=["context", "question", "chat_history"],
    template="""
    你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
    请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。
    
    背景资料：{context}
    
    问题：{question}

    对话历史：{chat_history}

    回答：
    """,
)

def format_docs(docs):
    formatted_docs = "\n\n".join(doc.page_content for doc in docs)
    return formatted_docs

def format_chat_history(chat_history):
    if not chat_history: return "暂无对话历史"

    formatted_chat_history =[]
    for message in chat_history:
        if isinstance(message, HumanMessage):
            formatted_chat_history.append(f"用户：{message.content}")
        elif isinstance(message, AIMessage):
            formatted_chat_history.append(f"助手：{message.content}")
        else:
            formatted_chat_history.append(f"消息：{message.content}")
            
    return "\n".join(formatted_chat_history)

llm = init_chat_model("qwen-plus",model_provider="openai")

rag_chain = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs, 
        "question": lambda x: x["question"],
        "chat_history": lambda x: format_chat_history(x["chat_history"]),
    }  
    | prompt 
    | llm
    | StrOutputParser()
    )

In [4]:
chat_history = []
question1="中国科学院国家天文台2023年部门预算总额是多少"
result = rag_chain.invoke({
    "question":question1,
    "chat_history": chat_history
})
print(result)

中国科学院国家天文台2023年部门预算总额是198,223.16万元。


In [5]:
chat_history.append(HumanMessage(content=question1))
chat_history.append(AIMessage(content=result))

In [6]:
chat_history = []
question2 = "该预算中，科学技术支出具体是多少？"
result2 = rag_chain.invoke({
    "question": question2,
    "chat_history": chat_history
})
print(result2)

该预算中，科学技术支出具体是 **117,981.03 万元**（即 2023 年初预算数）。  

依据背景资料：“2023 年初 ，科学技术支出预算数为 117,981.03 万元”。  
（注：表格中出现的“54,561.72”万元为“本年一般公共预算支出”合计数，属于执行或细化口径，而问题问的是“该预算中”的总科学技术支出，应以明确给出的年初预算数为准。）
